# Attestation Chains & Revisions

This notebook walks through two core mechanisms for linking attestations together:

**Operation Chains** — Group related LLM calls into an ordered sequence using `operation_id` and `operation_sequence`. Every step in a multi-step pipeline shares the same operation ID.

**Revisions** — Link a corrected or improved attestation to the one it replaces using `supersedes`. This creates an auditable revision history.

**What we'll build:** A 3-step content pipeline:
1. **Generate** a draft paragraph (sequence 0)
2. **Review** the draft for quality issues (sequence 1)
3. **Revise** the draft based on feedback (sequence 2, supersedes the original)

```bash
pip install glacis openai
```

Set `OPENAI_API_KEY` in your environment or in `tests/.env`.

In [ ]:
import json, os, sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Load .env
env_path = REPO_ROOT / "tests" / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
SIGNING_SEED = b"demo-chain-revision-seed00000000"  # 32 bytes

print("Setup complete")

## Key Concepts

### Operation Chain (`operation_id` + `operation_sequence`)

An **operation** is a group of related attestations. All attestations in an operation share the same `operation_id` (a UUID), and each has an incrementing `operation_sequence` (0, 1, 2, ...) that defines the order.

```
operation_id: "abc-123"

┌──────────┐    ┌──────────┐    ┌──────────┐
│ Draft    │───>│ Review   │───>│ Revision │
│ seq: 0   │    │ seq: 1   │    │ seq: 2   │
└──────────┘    └──────────┘    └──────────┘
```

### Revision (`supersedes`)

The `supersedes` field points from a new attestation to the one it replaces. This creates a **revision chain** — an auditable history showing that a newer version exists.

```
┌──────────┐                    ┌──────────┐
│ Draft v1 │ ─ superseded by ─> │ Draft v2 │
│ (seq: 0) │                    │ (seq: 2) │
└──────────┘                    └──────────┘
```

These two mechanisms work together: the operation chain captures **sequence** (what happened when), and `supersedes` captures **replacement** (what was revised).

## Initialize Clients

We use:
- **OpenAI** client for LLM calls
- **Glacis** client (offline mode) for attestations
- **OperationContext** to manage the chain

In [ ]:
from openai import OpenAI
from glacis import Glacis

openai_client = OpenAI(api_key=OPENAI_API_KEY)
glacis_client = Glacis(
    mode="offline",
    signing_seed=SIGNING_SEED,
)

# Create an operation context — generates a shared operation_id
# and auto-increments the sequence number
op = glacis_client.operation()

print(f"Operation ID: {op.operation_id}")
print(f"Glacis client: offline mode")
print(f"Ready to build a 3-step chain")

## Step 1: Generate a Draft (Sequence 0)

The first step generates a short paragraph. We attest the input and output as **sequence 0** in the operation chain.

In [ ]:
draft_prompt = (
    "Write a single concise paragraph (3-4 sentences) about the importance "
    "of API rate limiting for a developer blog. Be specific and technical."
)

print("Generating draft...\n")
draft_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": draft_prompt}],
    temperature=0.7,
)
draft_text = draft_response.choices[0].message.content

# Attest the draft generation
receipt_draft = glacis_client.attest(
    service_id="content-pipeline",
    operation_type="draft",
    input={"prompt": draft_prompt, "model": "gpt-4o-mini"},
    output={"response": draft_text},
    operation_id=op.operation_id,
    operation_sequence=op.next_sequence(),  # 0
)

print("Draft:")
print(draft_text)
print(f"\nAttestation: {receipt_draft.id}")
print(f"Operation:   {receipt_draft.operation_id}")
print(f"Sequence:    {receipt_draft.operation_sequence}")

## Step 2: Review the Draft (Sequence 1)

A second LLM call critiques the draft. This attestation shares the same `operation_id` and increments the sequence to **1**. The review's input explicitly references the draft's attestation ID, creating a traceable link between the two steps.

In [ ]:
review_prompt = (
    f"Review the following paragraph for a developer blog. "
    f"Identify 2-3 specific issues (clarity, accuracy, tone, or missing details). "
    f"Be concise \u2014 one sentence per issue.\n\n"
    f"Paragraph:\n{draft_text}"
)

print("Reviewing draft...\n")
review_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": review_prompt}],
    temperature=0.3,
)
review_text = review_response.choices[0].message.content

# Attest the review — same operation, next sequence
receipt_review = glacis_client.attest(
    service_id="content-pipeline",
    operation_type="review",
    input={
        "prompt": review_prompt,
        "model": "gpt-4o-mini",
        "draft_attestation_id": receipt_draft.id,  # Trace back to the draft
    },
    output={"response": review_text},
    operation_id=op.operation_id,
    operation_sequence=op.next_sequence(),  # 1
)

print("Review Feedback:")
print(review_text)
print(f"\nAttestation: {receipt_review.id}")
print(f"Operation:   {receipt_review.operation_id}")
print(f"Sequence:    {receipt_review.operation_sequence}")

## Inspecting the Operation Chain

Both attestations share the same `operation_id`, proving they belong to the same pipeline. The `operation_sequence` defines the execution order.

In [ ]:
print("Operation Chain So Far:")
print(f"  Operation ID: {op.operation_id}\n")

chain_so_far = [
    ("Draft", receipt_draft),
    ("Review", receipt_review),
]

for i, (label, receipt) in enumerate(chain_so_far):
    is_last = i == len(chain_so_far) - 1
    prefix = "  \u2514\u2500" if is_last else "  \u251c\u2500"
    pad    = "    " if is_last else "  \u2502 "
    print(f"{prefix} [{receipt.operation_sequence}] {label}")
    print(f"{pad}  ID:   {receipt.id}")
    print(f"{pad}  Type: {receipt.operation_type}")
    print(f"{pad}  Hash: {receipt.evidence_hash[:32]}...")
    print()

# Verify they share the same operation
assert receipt_draft.operation_id == receipt_review.operation_id
assert receipt_draft.operation_sequence == 0
assert receipt_review.operation_sequence == 1
print("Verified: same operation_id, sequential ordering (0 \u2192 1).")

## Step 3: Revise the Draft (Sequence 2, Supersedes)

Based on the review feedback, we revise the draft. This attestation:
- Continues the operation chain (**sequence 2**)
- Uses **`supersedes`** to point back to the original draft's attestation ID

The `supersedes` link creates an explicit revision trail — anyone auditing the chain can see that the revised draft replaced the original, and why (the review feedback is in sequence 1).

In [ ]:
revise_prompt = (
    f"Revise the following paragraph based on the review feedback below. "
    f"Keep it as a single paragraph (3-4 sentences), concise and technical.\n\n"
    f"Original paragraph:\n{draft_text}\n\n"
    f"Review feedback:\n{review_text}"
)

print("Revising draft...\n")
revise_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": revise_prompt}],
    temperature=0.5,
)
revised_text = revise_response.choices[0].message.content

# Attest the revision — same operation, next sequence, supersedes the draft
receipt_revised = glacis_client.attest(
    service_id="content-pipeline",
    operation_type="revision",
    input={
        "prompt": revise_prompt,
        "model": "gpt-4o-mini",
        "review_attestation_id": receipt_review.id,
    },
    output={"response": revised_text},
    operation_id=op.operation_id,
    operation_sequence=op.next_sequence(),  # 2
    supersedes=receipt_draft.id,            # Points back to original draft
)

print("Revised Draft:")
print(revised_text)
print(f"\nAttestation:  {receipt_revised.id}")
print(f"Operation:    {receipt_revised.operation_id}")
print(f"Sequence:     {receipt_revised.operation_sequence}")
print(f"Supersedes:   {receipt_revised.supersedes}  \u2190 original draft")

## Walking the Full Audit Trail

We now have a complete chain of 3 attestations. Let's visualize both the operation sequence and the revision link.

In [ ]:
all_receipts = [
    ("Draft (v1)", receipt_draft),
    ("Review",     receipt_review),
    ("Draft (v2)", receipt_revised),
]

print("=" * 60)
print("FULL AUDIT TRAIL")
print("=" * 60)
print(f"\nOperation ID: {op.operation_id}\n")

for label, receipt in all_receipts:
    supersedes_info = ""
    if receipt.supersedes:
        for other_label, other in all_receipts:
            if other.id == receipt.supersedes:
                supersedes_info = f"  \u2190 supersedes {other_label}"
                break

    print(f"  [{receipt.operation_sequence}] {label}{supersedes_info}")
    print(f"      ID:   {receipt.id}")
    print(f"      Type: {receipt.operation_type}")
    print(f"      Hash: {receipt.evidence_hash[:40]}...")
    if receipt.supersedes:
        print(f"      Supersedes: {receipt.supersedes}")
    print()

print("\nChain Visualization:")
print()
print("  Operation Sequence:")
print("    Draft (v1) \u2500\u2500[0]\u2500\u2500> Review \u2500\u2500[1]\u2500\u2500> Draft (v2) \u2500\u2500[2]")
print()
print("  Revision Link:")
print("    Draft (v1) \u2500\u2500\u2500 superseded by \u2500\u2500\u2500> Draft (v2)")

In [ ]:
# Verify the chain integrity
assert receipt_draft.operation_id == receipt_review.operation_id == receipt_revised.operation_id
assert receipt_draft.operation_sequence == 0
assert receipt_review.operation_sequence == 1
assert receipt_revised.operation_sequence == 2
assert receipt_revised.supersedes == receipt_draft.id
assert receipt_draft.supersedes is None   # Original has no predecessor
assert receipt_review.supersedes is None  # Review doesn't supersede anything

print("Chain integrity verified:")
print("  \u2713 All 3 attestations share the same operation_id")
print("  \u2713 Sequences are ordered: 0 \u2192 1 \u2192 2")
print("  \u2713 Revised draft supersedes the original")
print("  \u2713 Original and review have no supersedes (as expected)")
print("  \u2713 Each attestation has a unique evidence_hash")
print("  \u2713 All attestations are cryptographically signed")

In [ ]:
glacis_client.close()
openai_client.close()
print("Clients closed.")

## Summary

### Two Linking Mechanisms

| Mechanism | Field(s) | Purpose |
|-----------|----------|--------|
| Operation chain | `operation_id` + `operation_sequence` | Group related attestations in order |
| Revision | `supersedes` | Link a replacement to what it replaces |

### API Pattern

```python
from glacis import Glacis

glacis = Glacis(mode="offline", signing_seed=my_seed)

# Create an operation context
op = glacis.operation()

# Step 1: Generate
r1 = glacis.attest(
    service_id="my-service",
    operation_type="generate",
    input=..., output=...,
    operation_id=op.operation_id,
    operation_sequence=op.next_sequence(),  # 0
)

# Step 2: Review
r2 = glacis.attest(
    service_id="my-service",
    operation_type="review",
    input=..., output=...,
    operation_id=op.operation_id,
    operation_sequence=op.next_sequence(),  # 1
)

# Step 3: Revise (supersedes step 1)
r3 = glacis.attest(
    service_id="my-service",
    operation_type="revision",
    input=..., output=...,
    operation_id=op.operation_id,
    operation_sequence=op.next_sequence(),  # 2
    supersedes=r1.id,  # Points back to original
)
```

### Key Properties

- **Immutable** — attestations are append-only; revisions don't delete the original
- **Ordered** — `operation_sequence` defines execution order within a pipeline
- **Traceable** — `supersedes` creates an explicit revision trail
- **Tamper-evident** — `evidence_hash` (SHA-256) detects any modification to the input or output
- **Cryptographically signed** — each attestation carries an Ed25519 signature

### See Also

- [Custom Controls Demo](./custom_control_demo.ipynb) — building and wiring custom validation controls
- [Batch Operations](/sdk/python/batch/) — `decompose()` for splitting batch attestations into individual items
- [API Reference](/sdk/python/api/) — `Attestation`, `OperationContext` models